<a href="https://colab.research.google.com/github/47096/tensorflow-demo/blob/main/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# predict MPG with neural networks using tensorflow

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import Sequential

random.seed(222)
np.random.seed(222)
tf.random.set_seed(222)


In [ ]:
# import the data
url = "data/auto-mpg.data"  # vendored UCI file

# the data does not contain a header
# so, we need to assign some column names
column_names = ["MPG", "Cylinders", "Displacement", "Horsepower", "Weight", "Acceleration", "Model Year", "Origin"]

df = pd.read_csv(url, names = column_names, sep = " ", comment = "\t", na_values = "?", skipinitialspace = True)


In [ ]:
# take a look at the data
df.head()


In [ ]:
# create a copy for backup
dataset = df.copy()

# glimpse the data
dataset.describe()


In [ ]:
# check for missing values
# Horsepower has 6 missing values 
dataset.isna().sum()


In [ ]:
# use the mean to impute the missing values in Horsepower
dataset['Horsepower'] = dataset['Horsepower'].fillna(dataset['Horsepower'].mean())

# check for missing values again
# all good! everything is zero
dataset.isna().sum()


In [ ]:
# count Origin
# Origin should not be numeric but categorical
dataset['Origin'].value_counts()


In [ ]:
# map Origin to a country
dataset['Origin'] = dataset['Origin'].map({1: 'USA', 2: 'Europe', 3: 'Japan'})
dataset.head()


In [ ]:
# create dummy variables using Origin
dataset = pd.get_dummies(dataset, columns=['Origin'], prefix='', prefix_sep='', dtype=float)
dataset.head()


In [ ]:
# split the data and create train and test sets
train_dataset = dataset.sample(frac = 0.8, random_state = 884)
test_dataset = dataset.drop(train_dataset.index)


In [ ]:
test_dataset.head()

In [ ]:
# separate features and labels
# drop MPG from both train and test sets
train_features = train_dataset.drop(["MPG"], axis = 1)
test_features = test_dataset.drop(["MPG"], axis = 1)

# create a label for each set
train_labels = train_dataset["MPG"]
test_labels = test_dataset["MPG"]


In [ ]:
# glimpse the train set
train_dataset.describe().transpose()


In [ ]:
# apply normalization using sklearn
# scale the data such that the mean will be 0 and the standard deviation will be 1
from sklearn.preprocessing import StandardScaler
feature_scaler = StandardScaler()
label_scaler = StandardScaler()

# fit on training Data
feature_scaler.fit(train_features.values)
label_scaler.fit(train_labels.values.reshape(-1, 1))

# transform both training and testing data
train_features = feature_scaler.transform(train_features.values)
test_features = feature_scaler.transform(test_features.values)

train_labels = label_scaler.transform(train_labels.values.reshape(-1, 1))
test_labels = label_scaler.transform(test_labels.values.reshape(-1, 1))


In [ ]:
# head of the normalised features
train_features[:5]

In [ ]:
# create a Deep Neural Network to train a regression model
model = Sequential([
    # 2 dense layers specified using Rectified Linear Unit activation function i.e. relu                    
    layers.Dense(32, activation='relu'), # hidden layer with 32 nodes
    layers.Dense(64, activation='relu'),
    layers.Dense(1) # add a dense layer with 1 neuron (output layer with one node)
])


In [ ]:
# define optimizer and loss function
model.compile(optimizer = "RMSProp", loss = "mse")


In [ ]:
# train (validate on a slice of train — keep test untouched until the end)
history = model.fit(
    x=train_features,
    y=train_labels,
    validation_split=0.2,
    epochs=100,
    verbose=0,
)


In [ ]:
def plot_loss(history):
    plt.plot(history.history['loss'], label='loss')
    plt.plot(history.history['val_loss'], label='val_loss')
    plt.ylim([0, 10])
    plt.xlabel('Epoch')
    plt.ylabel('Error (Loss)')
    plt.legend()
    plt.grid(True)

plot_loss(history)


In [ ]:
# evaluate on testing dataset
model.evaluate(test_features, test_labels)


In [ ]:
# make predictions
results = model.predict(test_features)

# decode using the scikit-learn object to get the result
decoded_result = label_scaler.inverse_transform(results.reshape(-1,1))


In [ ]:
decoded_result[:5]

In [ ]:
# convert array to dataframe
predictions = pd.DataFrame(decoded_result)


In [ ]:
predictions.head()

In [ ]:
predictions = predictions.rename(columns={0: 'pred'})


In [ ]:
predictions.head()

In [ ]:
# reset the index
test_dataset.reset_index(drop=True, inplace=True)

# concat 2 datasets
final = pd.concat([test_dataset, predictions], axis=1)


In [ ]:
final